In [ ]:

import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import HistGradientBoostingClassifier


current_dir = Path.cwd().resolve()

if current_dir.name == "experiments":
    project_root = current_dir.parent
elif (current_dir / "src").is_dir():
    project_root = current_dir
else:
    raise FileNotFoundError(
        "Khong tim thay thu muc goc DeTaiCuKy_V2 chua src"
    )

sys.path.insert(0, str(project_root))

print("Project root:", project_root)
print("Thu muc src:", (project_root / "src").exists())


from src.data import load_data
from src.config import RANDOM_STATE, CV_DIR
from src.evaluation import cv_evaluate
from src.pipelines import get_xgb
from src.experiment_log import ExperimentLogger

X_train, _, y_train, _, labels = load_data()

lg = ExperimentLogger("tuan")

ung_vien = {
    "hgb_N1_balanced_rep3": HistGradientBoostingClassifier(
        class_weight="balanced",
        early_stopping=False,
        random_state=RANDOM_STATE
    ),

    "xgb_default_rep3": get_xgb()["xgb_default"],
}

for ten, model in ung_vien.items():

    cv_df = cv_evaluate(
        model,
        X_train,
        y_train,
        ten,
        n_repeats=3
    )

    lg.log(
        model,
        cv_df,
        ten,
        strategy="CV lap 3x5",
        notes="kiem tra on dinh cho ung vien cuoi"
    )

    print(
        ten,
        "macro_f1", round(cv_df["macro_f1"].mean(), 4),
        "std", round(cv_df["macro_f1"].std(), 4),
        "recall_c11", round(cv_df["recall_c11"].mean(), 4)
    )

a = pd.read_csv(
    CV_DIR / "hgb_N1_balanced_rep3.csv"
)["macro_f1"].to_numpy()

b = pd.read_csv(
    CV_DIR / "xgb_default_rep3.csv"
)["macro_f1"].to_numpy()

d = b - a
k = len(d)

t_hc = d.mean() / np.sqrt(
    (1 / k + 0.25) * d.var(ddof=1)
)

p_value = 2 * stats.t.sf(
    abs(t_hc),
    df=k - 1
)

print("xgb tru hgb_N1:", round(d.mean(), 4))
print("t hieu chinh:", round(t_hc, 2))
print("p:", round(p_value, 3))


Project root: C:\Users\Lenovo\Downloads\ml\SGU26K2_MayHoc\DeTaiCuoiKy_V2
Thu muc src: True
hgb_N1_balanced_rep3 macro_f1 0.9661 std 0.0143 recall_c11 0.9657
xgb_default_rep3 macro_f1 0.9644 std 0.0175 recall_c11 0.96
xgb tru hgb_N1: -0.0018
t hieu chinh: -0.41
p: 0.688


In [ ]:
import pandas as pd
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import precision_recall_fscore_support
from src.config import RANDOM_STATE, CV_DIR

def per_class_report(model, X, y, class_labels=labels):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    pred = cross_val_predict(model, X, y, cv=cv)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y, pred, labels=list(range(12)), zero_division=0
    )

    report_df = pd.DataFrame({
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }, index=class_labels)

    return report_df.round(3)

rep_hgb = per_class_report(ung_vien["hgb_N1_balanced_rep3"], X_train, y_train)
rep_xgb = per_class_report(ung_vien["xgb_default_rep3"], X_train, y_train)

rep = pd.concat(
    {
        "hgb_N1_balanced": rep_hgb,
        "xgb_default": rep_xgb,
    },
    axis=1
)

output_path = CV_DIR / "finalists_per_class.csv"
rep.to_csv(output_path)

rep